# zhjudge：把训练好的模型发布到 Hugging Face

不重新训练：直接拿某次 Kaggle 训练版本输出里的 `outputs/model/<名字>/`，用仓库当前的代码重新生成模型卡，检查结构和许可后上传到你的 Hugging Face 仓库。默认先建**私有**仓库，你在网页上检查无误后再改成公开。

> 独立开源项目，与 TypeSafe（Jev）和 Convai（Laya）没有任何隶属、合作或背书关系；提及其名称只为说明模型类别（Jev 式类型化决策）和 Laya 检查点格式。不调用 Jev 的接口，也从不用它的输出训练。
> Independent open-source project, not affiliated with or endorsed by TypeSafe (Jev) or Convai (Laya); their names only identify the model category (Jev-style typed decisions) and the Laya checkpoint format. Jev's API is never called and its outputs are never trained on.

**开始前**（逐步说明见仓库 README 的「发布到 Hugging Face」）
1. 右侧 **Session options**：**Accelerator** 选 **None**（不需要 GPU），**Internet** 打开。
2. 右侧 **Input → Add Input**：选 **Your Work → Notebooks**，挂上那次训练的笔记本版本输出（例如跑 v1b 的那个版本）。
3. **Add-ons → Secrets**：添加 `HF_TOKEN`（Hugging Face 的 **Write** 令牌）并勾选本笔记本。仓库是公开的，克隆代码不需要 GitHub 令牌；只有用私有仓库时才另外需要 `GITHUB_TOKEN`（和训练笔记本同一个）。令牌只在内存里使用，不写进文件、不打印。
4. 在下面第 1 个单元格填 `HF_REPO_ID`，然后 **Run All**（几分钟）。

**English**: attach the training version's output as Input, add the Secret `HF_TOKEN` (a Hugging Face *write* token; `GITHUB_TOKEN` only for a private repo), set `HF_REPO_ID` below and Run All (no GPU needed). The repo is created private; make it public on the Hub after checking the page.

In [ ]:
# 1) 设置 / settings
REPO = "TerryG907/zhjudge"           # GitHub 仓库（公开；私有副本需要 Secret GITHUB_TOKEN）/ repo
BRANCH = "main"                      # 用哪个分支的代码生成模型卡 / code used for the model card
CONFIG = "configs/train_v1b.yaml"    # 与要发布的那次训练相同的配置 / the config the model was trained with
HF_REPO_ID = ""                      # 必填：你的用户名/模型名，例如 "yourname/zh-decision-mmbert-v1b"（最好不带 jev）
PUBLIC = False                       # False = 先建私有仓库，检查页面后到仓库 Settings 改成 Public
MODEL_NAME = ""                      # 留空 = 自动找 Input 里的导出目录；找到多个时填名字，如 "zhjudge-mmbert-base-v1b"

import os, subprocess, shutil, sys, base64
from pathlib import Path
if not HF_REPO_ID or "/" not in HF_REPO_ID:
    raise RuntimeError("先在上面填 HF_REPO_ID，格式是 你的用户名/模型名")

In [ ]:
# 2) 克隆代码：公开仓库不需要令牌；私有仓库用 Kaggle Secret GITHUB_TOKEN（不写进任何文件、不打印）/ clone the code
from kaggle_secrets import UserSecretsClient


def secret(name):
    try:
        return (UserSecretsClient().get_secret(name) or "").strip()
    except Exception:  # noqa: BLE001  (never print the exception)
        return ""


REPO_DIR = "/tmp/zhjudge"
_tok = secret("GITHUB_TOKEN")
_hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode() if _tok else ""
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if _tok:
    _env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
shutil.rmtree(REPO_DIR, ignore_errors=True)
_r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR],
                    env=_env, capture_output=True, text=True)
_msg = (_r.stdout + _r.stderr).strip()
if _tok:
    _msg = _msg.replace(_tok, "***").replace(_hdr, "***")
print(_msg, "（Secret GITHUB_TOKEN）" if _tok else "（匿名克隆 / anonymous clone）")
del _tok, _hdr, _env
if _r.returncode != 0:
    raise RuntimeError("克隆失败：检查 Internet 是否打开、REPO 和 BRANCH 是否正确；私有仓库还要检查 Secret GITHUB_TOKEN 是否存在、"
                       "是否勾选了本笔记本、是否过期；公开仓库不需要令牌，勾选了过期的 GITHUB_TOKEN 也会导致克隆失败，取消勾选即可")

In [ ]:
# 3) 安装依赖（CPU 版 PyTorch 就够）/ install (CPU torch is enough)
os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/zhjudge-venv"
os.environ.pop("PYTHONPATH", None)  # Kaggle 的 sitecustomize 依赖我们环境里没有的包 / not for our venv
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV = shutil.which("uv") or os.path.expanduser("~/.local/bin/uv")
if "zhjudge.publish" not in Path(REPO_DIR, "src/zhjudge/cli.py").read_text(encoding="utf-8"):
    raise RuntimeError(f"分支 {BRANCH} 的代码还没有 zhjudge publish：请用包含它的分支")
r = subprocess.run(["bash", "scripts/run_all.sh"], cwd=REPO_DIR,
                   env=dict(os.environ, ZHJUDGE_SYNC_ONLY="1", UV=UV, TORCH_VARIANT="cpu"))
if r.returncode:
    raise RuntimeError("依赖安装失败，见上面的输出")

In [ ]:
# 4) 在 Input 里找导出的模型 / find the exported checkpoint among the inputs
found = {}
for cfg_file in sorted(Path("/kaggle/input").rglob("rl_agent_config.json")):
    d = cfg_file.parent
    if (d / "model.safetensors").exists() and (d / "tokenizer").is_dir() and d.parent.name in ("model", "export"):
        found.setdefault(d.name, []).append(d)
for name, dirs in found.items():
    print(f"  {name}: {dirs[0]}")
if MODEL_NAME:
    found = {k: v for k, v in found.items() if k == MODEL_NAME}
if not found:
    raise RuntimeError("Input 里没有找到导出的模型：右侧 Add Input 挂上那次训练的笔记本版本输出" +
                       (f"（或检查 MODEL_NAME = {MODEL_NAME!r}）" if MODEL_NAME else ""))
if len(found) > 1:
    raise RuntimeError(f"找到多个模型 {sorted(found)}：在第 1 个单元格里把 MODEL_NAME 设成要发布的那个")
EXPORT_DIR = sorted(found.values())[0][0]
# 优先用 outputs/model/ 下的那份 / prefer the copy under outputs/model/
EXPORT_DIR = next((d for d in sorted(found.values())[0] if d.parent.name == "model"), EXPORT_DIR)
print("要发布 / publishing:", EXPORT_DIR)

In [ ]:
# 5) 生成模型卡并上传（令牌只通过环境变量交给这一条命令）/ regenerate the card and upload
_hf = secret("HF_TOKEN")
if not _hf:
    raise RuntimeError("读不到 Secret HF_TOKEN：在 Hugging Face 的 Settings → Access Tokens 建一个 Write 令牌，"
                       "存为 Kaggle Secret HF_TOKEN 并勾选本笔记本")
cmd = [UV, "run", "--no-sync", "zhjudge", "publish", str(EXPORT_DIR), "--repo-id", HF_REPO_ID, "--config", CONFIG,
       "--out", "/tmp/publish"] + (["--public"] if PUBLIC else [])
r = subprocess.run(cmd, cwd=REPO_DIR, env=dict(os.environ, HF_TOKEN=_hf), capture_output=True, text=True)
print((r.stdout + r.stderr).replace(_hf, "***")[-4000:])
del _hf
if r.returncode:
    raise RuntimeError("上传失败，原因见上面的输出（常见：令牌不是 Write 权限、HF_REPO_ID 的用户名不是你的）")
print(f"\n完成：https://huggingface.co/{HF_REPO_ID}" + ("" if PUBLIC else "（私有；检查后到 Settings → Change repo visibility 改成 Public）"))